In [ ]:
!pip install -q ultralytics gdown

In [ ]:
import ultralytics
ultralytics.checks()

In [ ]:
!nvidia-smi

In [ ]:
DRONE2021_ZIP_LINK =
VISIO_MODEL_LINK = 
UAV300_MODEL_LINK = 

In [ ]:
!gdown --fuzzy "{DRONE2021_ZIP_LINK}" -O /kaggle/working/DroneDetection2021_YOLO_Unified.zip
!gdown --fuzzy "{VISIO_MODEL_LINK}" -O /kaggle/working/visiodect_best.pt
!gdown --fuzzy "{UAV300_MODEL_LINK}" -O /kaggle/working/uav300_best.pt

In [ ]:
!unzip -q /kaggle/working/DroneDetection2021_YOLO_Unified.zip -d /kaggle/working/drone2021_raw

In [ ]:
!find /kaggle/working/drone2021_raw -maxdepth 5 -type d | head -100

In [ ]:
from pathlib import Path

base = Path("/kaggle/working/drone2021_raw")

image_all_candidates = list(base.rglob("images/all"))
label_all_candidates = list(base.rglob("labels/all"))

print("Image all candidates:")
for p in image_all_candidates:
    print(p)

print("\nLabel all candidates:")
for p in label_all_candidates:
    print(p)

In [ ]:
images_dir = image_all_candidates[0]
labels_dir = label_all_candidates[0]

print("Using images:", images_dir)
print("Using labels:", labels_dir)

print("Images:", len(list(images_dir.glob("*"))))
print("Labels:", len(list(labels_dir.glob("*.txt"))))

In [ ]:
import shutil
import random
from pathlib import Path

output_path = Path("/kaggle/working/DroneDetection2021_split")

if output_path.exists():
    shutil.rmtree(output_path)

for split in ["train", "val", "test"]:
    (output_path / "images" / split).mkdir(parents=True, exist_ok=True)
    (output_path / "labels" / split).mkdir(parents=True, exist_ok=True)

image_files = []
for ext in ["*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp"]:
    image_files.extend(images_dir.glob(ext))

print("Total images:", len(image_files))

random.seed(42)
random.shuffle(image_files)

train_ratio = 0.7
val_ratio = 0.2

train_end = int(len(image_files) * train_ratio)
val_end = int(len(image_files) * (train_ratio + val_ratio))

splits = {
    "train": image_files[:train_end],
    "val": image_files[train_end:val_end],
    "test": image_files[val_end:]
}

missing_labels = 0

for split, files in splits.items():
    for img_path in files:
        shutil.copy(img_path, output_path / "images" / split / img_path.name)

        label_path = labels_dir / f"{img_path.stem}.txt"
        dst_label_path = output_path / "labels" / split / f"{img_path.stem}.txt"

        if label_path.exists():
            shutil.copy(label_path, dst_label_path)
        else:
            dst_label_path.write_text("")
            missing_labels += 1

print("Dataset split completed")
print("Missing labels:", missing_labels)

for split in ["train", "val", "test"]:
    imgs = list((output_path / "images" / split).glob("*"))
    labels = list((output_path / "labels" / split).glob("*.txt"))
    print(split, "images:", len(imgs), "labels:", len(labels))

In [ ]:
from collections import Counter

classes = Counter()

for label_file in (output_path / "labels" / "test").glob("*.txt"):
    for line in label_file.read_text().splitlines():
        parts = line.strip().split()
        if len(parts) >= 5:
            classes[parts[0]] += 1

print(classes)

In [ ]:
for split in ["train", "val", "test"]:
    for label_file in (output_path / "labels" / split).glob("*.txt"):
        new_lines = []
        for line in label_file.read_text().splitlines():
            parts = line.strip().split()
            if len(parts) >= 5:
                parts[0] = "0"
                new_lines.append(" ".join(parts))
        label_file.write_text("\n".join(new_lines))

print("Converted all labels to class 0 = drone")

In [ ]:
yaml_content = f"""
path: {output_path}

train: images/train
val: images/val
test: images/test

names:
  0: drone
"""

with open("/kaggle/working/drone2021.yaml", "w") as f:
    f.write(yaml_content)

print(open("/kaggle/working/drone2021.yaml").read())

In [ ]:
#visioDECT model
!yolo detect val \
  model=/kaggle/working/visiodect_best.pt \
  data=/kaggle/working/drone2021.yaml \
  imgsz=640 \
  split=test \
  device=0 \
  project=/kaggle/working/yolo_runs \
  name=cross_eval_train_visiodect_test_dronedetect2021

In [ ]:
#uav
!yolo detect val \
  model=/kaggle/working/uav300_best.pt \
  data=/kaggle/working/drone2021.yaml \
  imgsz=640 \
  split=test \
  device=0 \
  project=/kaggle/working/yolo_runs \
  name=cross_eval_train_uav300_test_dronedetect2021

In [ ]:
!yolo detect predict \
  model=/kaggle/working/visiodect_best.pt \
  source=/kaggle/working/DroneDetection2021_split/images/test \
  imgsz=640 \
  conf=0.25 \
  device=0 \
  save=True \
  project=/kaggle/working/yolo_runs \
  name=cross_pred_train_visiodect_test_dronedetect2021

In [ ]:
!yolo detect predict \
  model=/kaggle/working/uav300_best.pt \
  source=/kaggle/working/DroneDetection2021_split/images/test \
  imgsz=640 \
  conf=0.25 \
  device=0 \
  save=True \
  project=/kaggle/working/yolo_runs \
  name=cross_pred_train_uav300_test_dronedetect2021

In [ ]:
!zip -r /kaggle/working/dronedetect2021_cross_tests.zip \
  /kaggle/working/yolo_runs/cross_eval_train_visiodect_test_dronedetect2021 \
  /kaggle/working/yolo_runs/cross_eval_train_uav300_test_dronedetect2021 \
  /kaggle/working/yolo_runs/cross_pred_train_visiodect_test_dronedetect2021 \
  /kaggle/working/yolo_runs/cross_pred_train_uav300_test_dronedetect2021

In [ ]:
!ls -lh /kaggle/working/dronedetect2021_cross_tests.zip

In [ ]:
!curl https://rclone.org/install.sh | sudo bash
!rclone version

In [ ]:
!rclone listremotes

In [ ]:
from pathlib import Path

TOKEN = 
config_dir = Path("/root/.config/rclone")
config_dir.mkdir(parents=True, exist_ok=True)

config_text = f"""
[gdrive]
type = drive
scope = drive
token = {TOKEN}
"""

config_path = config_dir / "rclone.conf"
config_path.write_text(config_text)

print("rclone config written to:", config_path)

In [ ]:
!rclone listremotes

In [ ]:
!rclone lsd gdrive:

In [ ]:
!ls -lh /kaggle/working/dronedetect2021_cross_tests.zip

In [ ]:
!rclone copy \
  /kaggle/working/dronedetect2021_cross_tests.zip \
  gdrive: \
  --progress

In [ ]:
from pathlib import Path

images_dir = Path("/kaggle/working/drone2021_raw/DroneDetection2021_YOLO_Unified/Drones/images/all")
labels_dir = Path("/kaggle/working/drone2021_raw/DroneDetection2021_YOLO_Unified/Drones/labels/all")

print("Images dir:", images_dir)
print("Labels dir:", labels_dir)

print("Images:", len(list(images_dir.glob("*"))))
print("Labels:", len(list(labels_dir.glob("*.txt"))))

In [ ]:
import shutil
import random
from pathlib import Path

output_path = Path("/kaggle/working/DroneDetection2021_Drones_split")

if output_path.exists():
    shutil.rmtree(output_path)

for split in ["train", "val", "test"]:
    (output_path / "images" / split).mkdir(parents=True, exist_ok=True)
    (output_path / "labels" / split).mkdir(parents=True, exist_ok=True)

image_files = []
for ext in ["*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp"]:
    image_files.extend(images_dir.glob(ext))

print("Total drone images:", len(image_files))

random.seed(42)
random.shuffle(image_files)

train_ratio = 0.7
val_ratio = 0.2

train_end = int(len(image_files) * train_ratio)
val_end = int(len(image_files) * (train_ratio + val_ratio))

splits = {
    "train": image_files[:train_end],
    "val": image_files[train_end:val_end],
    "test": image_files[val_end:]
}

missing_labels = 0

for split, files in splits.items():
    for img_path in files:
        shutil.copy(img_path, output_path / "images" / split / img_path.name)

        label_path = labels_dir / f"{img_path.stem}.txt"
        dst_label_path = output_path / "labels" / split / f"{img_path.stem}.txt"

        if label_path.exists():
            shutil.copy(label_path, dst_label_path)
        else:
            dst_label_path.write_text("")
            missing_labels += 1

print("Split completed.")
print("Missing labels:", missing_labels)

for split in ["train", "val", "test"]:
    imgs = list((output_path / "images" / split).glob("*"))
    labels = list((output_path / "labels" / split).glob("*.txt"))

    boxes = 0
    empty = 0
    for lf in labels:
        lines = [x for x in lf.read_text().splitlines() if x.strip()]
        boxes += len(lines)
        if len(lines) == 0:
            empty += 1

    print(split, "images:", len(imgs), "labels:", len(labels), "boxes:", boxes, "empty labels:", empty)

In [ ]:
from collections import Counter

classes = Counter()

for label_file in (output_path / "labels" / "test").glob("*.txt"):
    for line in label_file.read_text().splitlines():
        parts = line.strip().split()
        if len(parts) >= 5:
            classes[parts[0]] += 1

print(classes)

In [ ]:
for split in ["train", "val", "test"]:
    for label_file in (output_path / "labels" / split).glob("*.txt"):
        new_lines = []
        for line in label_file.read_text().splitlines():
            parts = line.strip().split()
            if len(parts) >= 5:
                parts[0] = "0"
                new_lines.append(" ".join(parts))
        label_file.write_text("\n".join(new_lines))

print("Converted Drones-only subset labels to class 0.")

In [ ]:
yaml_content = f"""
path: {output_path}

train: images/train
val: images/val
test: images/test

names:
  0: drone
"""

with open("/kaggle/working/drone2021_drones_only.yaml", "w") as f:
    f.write(yaml_content)

print(open("/kaggle/working/drone2021_drones_only.yaml").read())

In [ ]:
!yolo detect val \
  model=/kaggle/working/visiodect_best.pt \
  data=/kaggle/working/drone2021_drones_only.yaml \
  imgsz=640 \
  split=test \
  device=0 \
  project=/kaggle/working/yolo_runs \
  name=cross_eval_train_visiodect_test_dronedetect2021_drones_only

In [ ]:
!yolo detect val \
  model=/kaggle/working/uav300_best.pt \
  data=/kaggle/working/drone2021_drones_only.yaml \
  imgsz=640 \
  split=test \
  device=0 \
  project=/kaggle/working/yolo_runs \
  name=cross_eval_train_uav300_test_dronedetect2021_drones_only

In [ ]:
!yolo detect predict \
  model=/kaggle/working/visiodect_best.pt \
  source=/kaggle/working/DroneDetection2021_Drones_split/images/test \
  imgsz=640 \
  conf=0.25 \
  device=0 \
  save=True \
  project=/kaggle/working/yolo_runs \
  name=cross_pred_train_visiodect_test_dronedetect2021_drones_only

In [ ]:
!yolo detect predict \
  model=/kaggle/working/uav300_best.pt \
  source=/kaggle/working/DroneDetection2021_Drones_split/images/test \
  imgsz=640 \
  conf=0.25 \
  device=0 \
  save=True \
  project=/kaggle/working/yolo_runs \
  name=cross_pred_train_uav300_test_dronedetect2021_drones_only

In [ ]:
!zip -r /kaggle/working/dronedetect2021_drones_only_cross_tests.zip \
  /kaggle/working/yolo_runs/cross_eval_train_visiodect_test_dronedetect2021_drones_only \
  /kaggle/working/yolo_runs/cross_eval_train_uav300_test_dronedetect2021_drones_only \
  /kaggle/working/yolo_runs/cross_pred_train_visiodect_test_dronedetect2021_drones_only \
  /kaggle/working/yolo_runs/cross_pred_train_uav300_test_dronedetect2021_drones_only

In [ ]:
from pathlib import Path

TOKEN = 
config_dir = Path("/root/.config/rclone")
config_dir.mkdir(parents=True, exist_ok=True)

config_text = f"""
[gdrive]
type = drive
scope = drive
token = {TOKEN}
"""

config_path = config_dir / "rclone.conf"
config_path.write_text(config_text)

print("rclone config written to:", config_path)

In [ ]:
!rclone copy \
  /kaggle/working/dronedetect2021_drones_only_cross_tests.zip \
  gdrive: \
  --progress